![Degirum banner](https://raw.githubusercontent.com/DeGirum/PySDKExamples/main/images/degirum_banner.png)
## This notebook is an example of how to pipeline a single model and a combined model. 
This notebook is an example of how to use DeGirum PySDK to do AI inference of a graphical file using 
three AI models: face detection, age estimation, and gender classification. The face detection model 
is run on the image and the results are then processed by both the age estimation and the gender
classification model in parallel, one face at a time. Combined result is then displayed.
This script uses PIL as image processing backend.

This script works with the following inference options:

1. Run inference on DeGirum Cloud Platform;
2. Run inference on DeGirum AI Server deployed on a localhost or on some computer in your LAN or VPN;
3. Run inference on DeGirum ORCA accelerator directly installed on your computer.

To try different options, you need to specify the appropriate `hw_location` option. 

When running this notebook locally, you need to specify your cloud API access token in the [env.ini](../../env.ini) file, located in the same directory as this notebook.

When running this notebook in Google Colab, the cloud API access token should be stored in a user secret named `DEGIRUM_CLOUD_TOKEN`.

In [1]:
# make sure degirum-tools package is installed
!pip show degirum-tools || pip install degirum-tools

Name: degirum_tools
Version: 0.5.2
Summary: Tools for PySDK
Home-page: 
Author: DeGirum
Author-email: 
License: 
Location: /home/degirum/miniconda3/lib/python3.11/site-packages
Requires: degirum, ffmpegcv, ipython, numpy, opencv-python, pafy, pillow, psutil, pycocotools, pyyaml, requests, scipy, youtube-dl
Required-by: 


#### Specify where do you want to run your inferences

In [2]:
# hw_location: where you want to run inference.
#     Use "@cloud" to use DeGirum cloud.
#     Use "@local" to run on local machine.
#     Use an IP address for AI server inference.#
# face_model_zoo_url: URL/path for the face model zoo.
#     Use cloud_zoo_url for @cloud, @local, and AI server inference options.
#     Use '' for an AI server serving models from a local folder.
#     Use a path to a JSON file for a single model zoo in case of @local inference.#
# face_model_name: name of the model for face detection.
# age_model_zoo_url: URL/path for the age model zoo.
# age_model_name: name of the model for age estimation.
# gender_model_zoo_url: URL/path for the gender model zoo.
# gender_model_name: name of the model for gender detection.
# video_source: video source for inference
#     camera index for local camera
#     URL of RTSP stream
#     URL of YouTube Video
#     path to video file (mp4 etc)
hw_location = "@cloud"
face_model_zoo_url = "https://cs.degirum.com/degirum/ultralytics_v6"
face_model_name = "yolov8n_relu6_face--640x640_quant_n2x_orca1_1"
age_model_zoo_url = "https://cs.degirum.com/degirum/Yolov8-reg"
age_model_name = "yolov8n_relu6_age--256x256_quant_n2x_orca1_3"
gender_model_zoo_url = "https://cs.degirum.com/degirum/openvino"
gender_model_name = "yolov8n_relu6_gender--256x256_quant_openvino_cpu_1"
video_source = (
    "https://raw.githubusercontent.com/DeGirum/PySDKExamples/main/images/faces_and_gender.mp4"
)              

#### The rest of the cells below should run without any modifications

In [3]:
import degirum as dg
import degirum_tools

class CombiningCompoundModel2(degirum_tools.CombiningCompoundModel):
 
    # fallback all model-like attributes to the second model
    def __getattr__(self, attr):
        return getattr(self.model2, attr)
    
    @property
    def non_blocking_batch_predict(self):
        return self.model2.non_blocking_batch_predict
    
    @non_blocking_batch_predict.setter
    def non_blocking_batch_predict(self, val: bool):
        self.model1.non_blocking_batch_predict = val
        self.model2.non_blocking_batch_predict = val
 
    def transform_result2(self, result2):
        result2.results[0][
            "label"
        ] = f"{result2.results[0]['label']}: {result2.info.result1.results[0]['label']} {result2.info.result1.results[0]['score']}"
        return result2

# Connect to AI inference engine 
face_zoo = dg.connect(hw_location, face_model_zoo_url, degirum_tools.get_token())
age_zoo = dg.connect(hw_location, age_model_zoo_url, degirum_tools.get_token())
gender_zoo = dg.connect(hw_location, gender_model_zoo_url, degirum_tools.get_token())

# Load models
face_model = face_zoo.load_model(face_model_name, overlay_line_width=2, input_letterbox_fill_color=(114, 114, 114))
age_model = age_zoo.load_model(age_model_name)
gender_model = gender_zoo.load_model(gender_model_name)
# Create a combined model for age estimation and gender detection
age_gender_combined_model = CombiningCompoundModel2(age_model, gender_model)

# Create a compound cropping model with 50% crop extent
crop_model = degirum_tools.CroppingAndClassifyingCompoundModel(
    face_model, age_gender_combined_model, 50.0
)

# Detect faces and genders
with degirum_tools.Display("Faces, Age and Gender") as display:
    for age_gender_results in degirum_tools.predict_stream(crop_model, video_source):
        display.show(age_gender_results)

AttributeError: 'CombiningCompoundModel' object has no attribute 'image_backend'

In [ ]:
age_model.non_blocking_batch_predict

In [ ]:
age_gender_combined_model.model2.non_blocking_batch_predict

In [ ]:
age_model